In [ ]:
import os
import json
from typing import List
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login



# Authenticate with Hugging Face using your access token
login("your_hugging_face_token")  # Replace with your actual token


# Configuration
MODEL_ID = "meta-llama/Llama-2-7b-chat-hf"  # requires appropriate access
OUTPUT_DIR = "model"
BATCH_SIZE = 4                 # adjust per VRAM
MAX_LENGTH = 1024              # truncate to control memory

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
# Ensure a pad token for batching; Llama's pad_token is typically eos
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# device_map="auto" to spread across available devices; set torch_dtype to save memory
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    output_hidden_states=True,
    device_map="auto",
)

model.eval()


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import numpy as np
import os
import json
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib
import matplotlib.pyplot as plt
import gc

# ---------------- CONFIG ----------------

PROMPT_JSON_FILE = "/project/thesis_work/causal_probing_llms/data/empathy_prompts.json"  # Adjust path to your local file
OUTPUT_DIR = "/project/thesis_work/causal_probing_llms/concepts/empathy/gcav_results_toxicity_run2"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ---------------- LOAD PROMPTS ----------------
def load_prompts_from_json(file_path):
    """Reads positive and negative prompts from a JSON file."""
    print(f"Reading prompts from: {file_path}")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Prompt file not found: {file_path}")

    with open(file_path, "r") as f:
        data = json.load(f)

    positive_prompts = data.get("positive", [])
    negative_prompts = data.get("negative", [])

    print(f"Loaded {len(positive_prompts)} positive and {len(negative_prompts)} negative prompts.")
    return positive_prompts, negative_prompts


positive_prompts, negative_prompts = load_prompts_from_json(PROMPT_JSON_FILE)


# ---------------- FEATURE EXTRACTION FUNCTION ----------------
def extract_token_hidden(prompts, label, layer_idx):
    """Extracts hidden states for all tokens in the prompts at a given layer."""
    X, y = [], []
    for prompt in tqdm(prompts, desc=f"[Layer {layer_idx}] Extracting {label} samples"):
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)
            hidden = outputs.hidden_states[layer_idx].squeeze(0).cpu().float().numpy()

        X.append(hidden)
        y.extend([1 if label == "positive" else 0] * hidden.shape[0])

        # Memory cleanup
        del inputs, outputs, hidden
        gc.collect()
        torch.cuda.empty_cache()

    return np.vstack(X), np.array(y)


# ---------------- MAIN LOOP OVER LAYERS ----------------
num_layers = model.config.num_hidden_layers
layer_accuracies = []

# for layer_idx in range(num_layers):
for layer_idx in num_layers:
    print(f"\n===== Processing Layer {layer_idx} =====")

    # Extract features
    X_pos, y_pos = extract_token_hidden(positive_prompts, "positive", layer_idx)
    X_neg, y_neg = extract_token_hidden(negative_prompts, "negative", layer_idx)

    X = np.vstack((X_pos, X_neg))
    y = np.concatenate((y_pos, y_neg))

    print(f"Feature matrix for layer {layer_idx}: {X.shape}")

    # Train probe
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    layer_accuracies.append(acc)

    print(f"✅ Layer {layer_idx} probe accuracy: {acc:.4f}")

    # Save probe and CAV
    probe_path = os.path.join(OUTPUT_DIR, f"linear_probe_layer{layer_idx}.joblib")
    cav_path = os.path.join(OUTPUT_DIR, f"cav_layer{layer_idx}.npy")

    joblib.dump(clf, probe_path)
    concept_vector = clf.coef_[0] / np.linalg.norm(clf.coef_[0])
    np.save(cav_path, concept_vector)

    print(f"Saved probe → {probe_path}")
    print(f"Saved CAV   → {cav_path}")

    # Free up memory after each layer
    del X_pos, y_pos, X_neg, y_neg, X, y, X_train, X_test, y_train, y_test, clf
    gc.collect()
    torch.cuda.empty_cache()


# ---------------- VISUALIZE ACCURACY ----------------
plt.figure(figsize=(10, 6))
plt.plot(range(num_layers), layer_accuracies, marker='o', linestyle='-', linewidth=2)
plt.title("Linear Probe Accuracy per Layer (LLaMA-2 7B Chat)")
plt.xlabel("Layer Index")
plt.ylabel("Accuracy")
plt.grid(True)
plt.savefig(os.path.join(OUTPUT_DIR, "layerwise_probe_accuracy.png"))
plt.show()

# Optionally, save results
np.save(os.path.join(OUTPUT_DIR, "layer_accuracies.npy"), np.array(layer_accuracies))
print("\n✅ All layers processed. Results saved.")